### Deliverable 4 - INF201 

Submission deadline: 27th Nov 2026

Names: Sigurd Aleksander Ruud and Shem Abdul-Rahman


## Task 1: Temperatures

Create a `Temperature` class that supports Celsius ("C"), Fahrenheit ("F"), and Kelvin ("K"). This class should:
1. Be initialised with a value and a unit (case-insensitive), and raise a `ValueError` if the unit is not one of "C", "F" or "K", or if the temperature is below absolute zero (0 K).
2. Use a tolerance of 1e-9 for floating-point value comparisons
3. Expose `value` and `unit` as read-only attributes
4. Contain the following conversion methods:
    * `to_celsius(self) -> float`:
        * From Kelvin: $C = K - 273.15$
        * From Fahrenheit: $C = (F - 32) \times \frac{5}{9}$
    * `to_fahrenheit(self) -> float`:
        * From Celsius: $F = C \times \frac{9}{5} + 32$
        * From Kelvin: $F = (K - 273.15) \times \frac{9}{5} + 32$
    * `to_kelvin(self) -> float`:
        * From Celsius: $K = C + 273.15$
        * From Fahrenheit: $K = (F - 32) \times \frac{5}{9} + 273.15$
    * `convert_to(self, target_unit: str) -> Temperature`:
        * Returns a new `Temperature` instance converted to `target_unit` (case-insensitive), raising `ValueError` if `target_unit` is invalid.
5. Contain the following string representations:
    * `__repr__(self) -> str`: Returns an unambiguous representation, such as `Temperature(25.0, 'C')`.
    * `__str__(self) -> str`: Returns a human-readable string formatted to two decimal places, such as `25.00 °C`.
6. Contain all six comparisons (==, !=, <, <=, >, >=) so temperatures can be compared across different units (`Temperature(0, "C") == Temperature(32, "F")`). Should return `NotImplemented` if a `Temperature` instance is compared to a non-Temperature instance.

Write a `test_temperature.py` script that imports the `Temperature` class and uses pytest for the following tests:
1. Initialization: Default unit, case-insensitive units, and read-only behavior.
2. Validation: Invalid unit strings and temperatures below absolute zero in "C", "F", and "K".
3. Conversions: Test conversion accuracy across reference points:
    * Absolute zero ($-273.15^\circ\text{C} = -459.67^\circ\text{F} = 0\text{ K}$)
    * Celsius–Fahrenheit crossover ($-40^\circ\text{C} = -40^\circ\text{F} = 233.15\text{ K}$)
    * Freezing point of water ($0^\circ\text{C} = 32^\circ\text{F} = 273.15\text{ K}$)
    * Human body temperature ($37^\circ\text{C} = 98.6^\circ\text{F} = 310.15\text{ K}$)
    * Boiling point of water ($100^\circ\text{C} = 212^\circ\text{F} = 373.15\text{ K}$)
4. Round-trip conversions: Converting "C" -> "F" -> "K" -> "C" reproduces the original Temperature.
5. Cross-unit comparisons: Equality and ordering (<, <=, >, >=) across mixed units.
6. Formatting: Outputs of `str()` and `repr()`

## Task 1 - Temperature and test

In [6]:
import math


class Temperature:
    """Represent a temperature in Celsius, Fahrenheit, or Kelvin."""

    # Requirement 2: Use this tolerance when comparing floating-point values.
    TOLERANCE = 1e-9

    def __init__(self, value: float, unit: str = "C"):
        # Requirement 1: Accept a temperature value and a unit.
        # Convert the unit to uppercase so 'c', 'C', etc. are accepted.
        unit = unit.upper()

        # Requirement 1: Reject units other than C, F, and K.
        if unit not in ("C", "F", "K"):
            raise ValueError("Unit must be C, F, or K")

        # Convert to Kelvin temporarily to check the absolute-zero limit.
        if unit == "C":
            kelvin = value + 273.15
        elif unit == "F":
            kelvin = (value - 32) * 5 / 9 + 273.15
        else:
            kelvin = value
 
        # Requirement 1 (using requirement 2's tolerance):
        # Reject values that are more than 1e-9 K below absolute zero.
        if kelvin < -self.TOLERANCE:
            raise ValueError("Temperature cannot be below absolute zero")

        # Requirement 3: Store the data privately, behind read-only properties.
        self._value = value
        self._unit = unit

    @property
    def value(self) -> float:
        # Requirement 3: Allow reading, but not assigning, temperature.value.
        return self._value

    @property
    def unit(self) -> str:
        # Requirement 3: Allow reading, but not assigning, temperature.unit.
        return self._unit

    def to_celsius(self) -> float:
        # Requirement 4: Convert the stored temperature to Celsius.
        if self.unit == "C":
            return self.value
        if self.unit == "K":
            return self.value - 273.15
        return (self.value - 32) * 5 / 9

    def to_fahrenheit(self) -> float:
        # Requirement 4: Convert the stored temperature to Fahrenheit.
        if self.unit == "F":
            return self.value
        if self.unit == "C":
            return self.value * 9 / 5 + 32
        return (self.value - 273.15) * 9 / 5 + 32

    def to_kelvin(self) -> float:
        # Requirement 4: Convert the stored temperature to Kelvin.
        if self.unit == "K":
            return self.value
        if self.unit == "C":
            return self.value + 273.15
        return (self.value - 32) * 5 / 9 + 273.15
    def convert_to(self, target_unit: str) -> "Temperature":
        # Requirement 4: Convert to a requested unit, ignoring letter case.
        target_unit = target_unit.upper()
        if target_unit == "C":
            return Temperature(self.to_celsius(), "C")
        if target_unit == "F":
            return Temperature(self.to_fahrenheit(), "F")
        if target_unit == "K":
            return Temperature(self.to_kelvin(), "K")
        # Requirement 4: Invalid target units must raise ValueError.
        raise ValueError("Target unit must be C, F, or K")

    def __repr__(self) -> str:
        # Requirement 5: Provide an unambiguous, developer-friendly form.
        return f"Temperature({self.value!r}, {self.unit!r})"

    def __str__(self) -> str:
        # Requirement 5: Display two decimal places and the degree/unit symbol.
        return f"{self.value:.2f} °{self.unit}" if self.unit != "K" else f"{self.value:.2f} K"

    def _difference(self, other):
        # Requirement 6: Only compare Temperature objects.
        if not isinstance(other, Temperature):
            return NotImplemented
        # Convert both temperatures to Kelvin before comparing them.
        return self.to_kelvin() - other.to_kelvin()

    def __eq__(self, other):
        # Requirement 6: Equality across different units, within tolerance.
        difference = self._difference(other)
        if difference is NotImplemented:
            return NotImplemented
        return math.isclose(difference, 0.0, abs_tol=self.TOLERANCE, rel_tol=0)

    def __ne__(self, other):
        # Requirement 6: Inequality is the opposite of equality.
        result = self.__eq__(other)
        if result is NotImplemented:
            return NotImplemented
        return not result

    def __lt__(self, other):
        # Requirement 6: Strictly less than, allowing for tolerance.
        difference = self._difference(other)
        if difference is NotImplemented:
            return NotImplemented
        return difference < -self.TOLERANCE

    def __le__(self, other):
        # Requirement 6: Less than or approximately equal to.
        difference = self._difference(other)
        if difference is NotImplemented:
            return NotImplemented
        return difference <= self.TOLERANCE

    def __gt__(self, other):
        # Requirement 6: Strictly greater than, allowing for tolerance.
        difference = self._difference(other)
        if difference is NotImplemented:
            return NotImplemented
        return difference > self.TOLERANCE

    def __ge__(self, other):
        # Requirement 6: Greater than or approximately equal to.
        difference = self._difference(other)
        if difference is NotImplemented:
            return NotImplemented
        return difference >= -self.TOLERANCE


In [7]:
import sys
!{sys.executable} -m pytest test_temp.py -v

============================= test session starts =============================
platform win32 -- Python 3.15.2, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\shems\AppData\Local\Programs\Python\Python315\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\shems\OneDrive - Norwegian University of Life Sciences\Høst2026\INF201\Deliverable_4
plugins: platformdirs-4.14.0
collecting ... collected 22 items

test_temp.py::test_initialization PASSED                                 [  4%]
test_temp.py::test_read_only_attributes PASSED                           [  9%]
test_temp.py::test_invalid_units PASSED                                  [ 13%]
test_temp.py::test_below_absolute_zero[-273.16-C] PASSED                 [ 18%]
test_temp.py::test_below_absolute_zero[-459.68-F] PASSED                 [ 22%]
test_temp.py::test_below_absolute_zero[-0.01-K] PASSED                   [ 27%]
test_temp.py::test_reference_conversions[-273.15--459.67-0] PASSED       [ 31%]
test_temp.py::test_reference_conversions

## Task 2: Fractions

Create a `Fraction` class representing fractions. Both numerator and denominator should be integers, and the denominator should be stricktly positive (>0). The class should implement the following special methods (aka magic methods, dunder methods): 
* initialiser
* all comparisons (equal, not equal, smaller than, smaller than or equal, greater than, and greated than or equal)
* at least four other special methods

Write a test script to be used with `pytest` for testing the implemented methods. Include the output from testing in a Jupyter notebok.

In [3]:

import pytest
from fraction import Fraction


# ----------------------------------------------------------------------
# Initialiser
# ----------------------------------------------------------------------
def test_init_stores_values():
    f = Fraction(3, 4)
    assert f.numerator == 3
    assert f.denominator == 4


def test_init_default_denominator():
    f = Fraction(5)
    assert f.numerator == 5
    assert f.denominator == 1


def test_init_reduces_to_lowest_terms():
    f = Fraction(6, 8)
    assert (f.numerator, f.denominator) == (3, 4)


def test_init_negative_numerator():
    f = Fraction(-2, 4)
    assert (f.numerator, f.denominator) == (-1, 2)


def test_init_zero_numerator():
    f = Fraction(0, 5)
    assert (f.numerator, f.denominator) == (0, 1)


@pytest.mark.parametrize("denominator", [0, -1, -5])
def test_init_denominator_not_positive_raises(denominator):
    with pytest.raises(ValueError):
        Fraction(1, denominator)


@pytest.mark.parametrize("numerator, denominator", [
    (1.5, 2), (1, 2.0), ("1", 2), (1, "2"), (True, 2), (None, 1),
])
def test_init_non_integer_raises(numerator, denominator):
    with pytest.raises(TypeError):
        Fraction(numerator, denominator)


def test_attributes_are_read_only():
    f = Fraction(1, 2)
    with pytest.raises(AttributeError):
        f.numerator = 5
    with pytest.raises(AttributeError):
        f.denominator = 5


# ----------------------------------------------------------------------
# Comparisons
# ----------------------------------------------------------------------
def test_eq():
    assert Fraction(1, 2) == Fraction(2, 4)
    assert Fraction(2, 1) == 2
    assert not (Fraction(1, 2) == Fraction(1, 3))


def test_ne():
    assert Fraction(1, 2) != Fraction(1, 3)
    assert not (Fraction(1, 2) != Fraction(3, 6))


def test_lt():
    assert Fraction(1, 3) < Fraction(1, 2)
    assert Fraction(-1, 2) < Fraction(1, 3)
    assert not (Fraction(1, 2) < Fraction(1, 2))


def test_le():
    assert Fraction(1, 3) <= Fraction(1, 2)
    assert Fraction(1, 2) <= Fraction(2, 4)
    assert not (Fraction(2, 3) <= Fraction(1, 2))


def test_gt():
    assert Fraction(3, 4) > Fraction(2, 3)
    assert Fraction(1, 2) > 0
    assert not (Fraction(1, 2) > Fraction(1, 2))


def test_ge():
    assert Fraction(3, 4) >= Fraction(2, 3)
    assert Fraction(2, 4) >= Fraction(1, 2)
    assert not (Fraction(1, 3) >= Fraction(1, 2))


def test_compare_with_other_type():
    # __eq__ returns NotImplemented, so Python falls back to False
    assert (Fraction(1, 2) == "1/2") is False
    assert Fraction(1, 2) != 0.5
    assert Fraction(1, 2).__eq__("1/2") is NotImplemented
    with pytest.raises(TypeError):
        Fraction(1, 2) < "a"


def test_hash_equal_fractions():
    assert hash(Fraction(1, 2)) == hash(Fraction(2, 4))
    assert len({Fraction(1, 2), Fraction(2, 4), Fraction(3, 6)}) == 1


# ----------------------------------------------------------------------
# Arithmetic
# ----------------------------------------------------------------------
def test_add():
    assert Fraction(1, 2) + Fraction(1, 3) == Fraction(5, 6)
    assert Fraction(1, 2) + 1 == Fraction(3, 2)
    assert 1 + Fraction(1, 2) == Fraction(3, 2)


def test_sub():
    assert Fraction(1, 2) - Fraction(1, 3) == Fraction(1, 6)
    assert Fraction(1, 3) - Fraction(1, 2) == Fraction(-1, 6)
    assert 1 - Fraction(1, 4) == Fraction(3, 4)


def test_mul():
    assert Fraction(2, 3) * Fraction(3, 4) == Fraction(1, 2)
    assert Fraction(1, 3) * 3 == 1
    assert 2 * Fraction(1, 4) == Fraction(1, 2)


def test_truediv():
    assert Fraction(1, 2) / Fraction(1, 4) == 2
    assert Fraction(1, 2) / 2 == Fraction(1, 4)


def test_truediv_by_negative_keeps_denominator_positive():
    result = Fraction(1, 2) / Fraction(-1, 3)
    assert result == Fraction(-3, 2)
    assert result.denominator > 0


def test_truediv_by_zero_raises():
    with pytest.raises(ZeroDivisionError):
        Fraction(1, 2) / Fraction(0, 1)


def test_arithmetic_with_other_type_raises():
    with pytest.raises(TypeError):
        Fraction(1, 2) + "a"


# ----------------------------------------------------------------------
# Unary operators and conversions
# ----------------------------------------------------------------------
def test_neg():
    assert -Fraction(1, 2) == Fraction(-1, 2)
    assert -Fraction(-3, 4) == Fraction(3, 4)


def test_abs():
    assert abs(Fraction(-3, 4)) == Fraction(3, 4)
    assert abs(Fraction(3, 4)) == Fraction(3, 4)


def test_float():
    assert float(Fraction(1, 4)) == pytest.approx(0.25)
    assert float(Fraction(-1, 3)) == pytest.approx(-0.333333333)


# ----------------------------------------------------------------------
# String representations
# ----------------------------------------------------------------------
def test_str():
    assert str(Fraction(3, 4)) == "3/4"
    assert str(Fraction(-6, 8)) == "-3/4"


def test_repr():
    assert repr(Fraction(3, 4)) == "Fraction(3, 4)"
    # repr should be valid Python that recreates the same object
    assert eval(repr(Fraction(2, 6))) == Fraction(1, 3)